In [1]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="olist-pipeline.cg1gusiqwbyb.us-east-1.rds.amazonaws.com",
    database="postgres",
    user="postgres",
    password="Admin1234!"
)

query = """
SELECT order_id, COUNT(*) 
FROM olist_raw.order_reviews 
GROUP BY order_id 
HAVING COUNT(*) > 1
LIMIT 5
"""

df = pd.read_sql(query, conn)
print(df.shape)
print(df)
conn.close()

C:\Users\deepe\AppData\Local\Temp\ipykernel_28564\277150062.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


(5, 2)
                           order_id  count
0  0035246a40f520710769010f752e7507      2
1  013056cfe49763c6f66bda03396c5ee3      2
2  0176a6846bcb3b0d3aa3116a9a768597      2
3  02355020fd0a40a0d56df9f6ff060413      2
4  029863af4b968de1e5d6a82782e662f5      2


In [7]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="olist-pipeline.cg1gusiqwbyb.us-east-1.rds.amazonaws.com",
    database="postgres",
    user="postgres",
    password="Admin1234!"
)

query = """
WITH deduped_reviews AS (
    SELECT
        order_id,
        review_score,
        ROW_NUMBER() OVER (
            PARTITION BY order_id 
            ORDER BY review_creation_date DESC
        ) AS rn
    FROM olist_raw.order_reviews
)
SELECT
    o.order_id,
    c.customer_state,
    oi.price,
    t.product_category_name_english AS category,
    r.review_score,
    TO_DATE(o.order_purchase_timestamp, 'DD-MM-YYYY HH24:MI') AS order_date
FROM olist_raw.orders o
JOIN olist_raw.customers c ON o.customer_id = c.customer_id
JOIN olist_raw.order_items oi ON o.order_id = oi.order_id
JOIN olist_raw.products p ON oi.product_id = p.product_id
JOIN olist_raw.product_category_translation t ON p.product_category_name = t.product_category_name
JOIN deduped_reviews r ON o.order_id = r.order_id AND r.rn = 1
WHERE o.order_status = 'delivered'
"""

df = pd.read_sql(query, conn)
print(df.shape)
print(df.head())
df.to_csv(r"D:\dbt_projects\olist_pipeline\olist_dashboard_data.csv", index=False)
conn.close()

C:\Users\deepe\AppData\Local\Temp\ipykernel_28564\579858144.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


(107823, 6)
                           order_id customer_state   price         category  \
0  00010242fe8c5a6d1ba2dd792cb16214             RJ   58.90       cool_stuff   
1  00018f77f2f0320c557190d7a144bdd3             SP  239.90         pet_shop   
2  000229ec398224ef6ca0657da4fc703e             MG  199.00  furniture_decor   
3  00024acbcdf0a6daa1e931b038114c75             SP   12.99        perfumery   
4  00048cc3ae777c65dbb7d2a0634bc1ea             MG   21.90       housewares   

   review_score  order_date  
0             5  2017-09-13  
1             4  2017-04-26  
2             5  2018-01-14  
3             4  2018-08-08  
4             4  2017-05-15  


In [8]:
print(df[df.duplicated('order_id', keep=False)].sort_values('order_id').head(10))

                               order_id customer_state  price        category  \
9      0008288aa423d2a3f00fcb17cd7d8719             SP  49.90    garden_tools   
10     0008288aa423d2a3f00fcb17cd7d8719             SP  49.90    garden_tools   
53905  001ab0a7578dd66cd4b0a71f5b6e1e41             BA  24.89     electronics   
53904  001ab0a7578dd66cd4b0a71f5b6e1e41             BA  24.89     electronics   
53903  001ab0a7578dd66cd4b0a71f5b6e1e41             BA  24.89     electronics   
21     001d8f0e34a38c37f7dba2a37d4eba8b             SP  18.99   health_beauty   
22     001d8f0e34a38c37f7dba2a37d4eba8b             SP  18.99   health_beauty   
53919  002c9def9c9b951b1bec6d50753c9891             SP  78.00      housewares   
53918  002c9def9c9b951b1bec6d50753c9891             SP  78.00      housewares   
38     002f98c0f7efd42638ed6100ca699b42             RS   8.99  consoles_games   

       review_score  order_date  
9                 5  2018-02-13  
10                5  2018-02-13  
53905 

In [11]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="olist-pipeline.cg1gusiqwbyb.us-east-1.rds.amazonaws.com",
    database="postgres",
    user="postgres",
    password="Admin1234!"
)

query2 = """
SELECT order_id, product_id, COUNT(*) 
FROM olist_raw.order_items
GROUP BY order_id, product_id
HAVING COUNT(*) > 1
LIMIT 5
"""

df_check = pd.read_sql(query2, conn)
print(df_check)

C:\Users\deepe\AppData\Local\Temp\ipykernel_28564\1008084862.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_check = pd.read_sql(query2, conn)


                           order_id                        product_id  count
0  0008288aa423d2a3f00fcb17cd7d8719  368c6c730842d78016ad823897a372db      2
1  00143d0f86d6fbd9f9b38ab440ac16f5  e95ee6822b66ac6058e2e4aff656071a      3
2  001ab0a7578dd66cd4b0a71f5b6e1e41  0b0172eb0fd18479d29c3bc122c058c2      3
3  001d8f0e34a38c37f7dba2a37d4eba8b  e67307ff0f15ade43fcb6e670be7a74c      2
4  002c9def9c9b951b1bec6d50753c9891  2d9ff06c8870a518f5f6909774e140fb      2


In [13]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="olist-pipeline.cg1gusiqwbyb.us-east-1.rds.amazonaws.com",
    database="postgres",
    user="postgres",
    password="Admin1234!"
)

query = """
WITH deduped_reviews AS (
    SELECT
        order_id,
        review_score,
        ROW_NUMBER() OVER (
            PARTITION BY order_id 
            ORDER BY review_creation_date DESC
        ) AS rn
    FROM olist_raw.order_reviews
)
SELECT DISTINCT
    o.order_id,
    c.customer_state,
    oi.price,
    t.product_category_name_english AS category,
    r.review_score,
    TO_DATE(o.order_purchase_timestamp, 'DD-MM-YYYY HH24:MI') AS order_date
FROM olist_raw.orders o
JOIN olist_raw.customers c ON o.customer_id = c.customer_id
JOIN olist_raw.order_items oi ON o.order_id = oi.order_id
JOIN olist_raw.products p ON oi.product_id = p.product_id
JOIN olist_raw.product_category_translation t ON p.product_category_name = t.product_category_name
JOIN deduped_reviews r ON o.order_id = r.order_id AND r.rn = 1
WHERE o.order_status = 'delivered'
"""

df = pd.read_sql(query, conn)
print(df.shape)
print(df.head())
df.to_csv(r"D:\dbt_projects\olist_pipeline\olist_dashboard_data.csv", index=False)
conn.close()

C:\Users\deepe\AppData\Local\Temp\ipykernel_28564\885570458.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


(97041, 6)
                           order_id customer_state   price         category  \
0  00010242fe8c5a6d1ba2dd792cb16214             RJ   58.90       cool_stuff   
1  00018f77f2f0320c557190d7a144bdd3             SP  239.90         pet_shop   
2  000229ec398224ef6ca0657da4fc703e             MG  199.00  furniture_decor   
3  00024acbcdf0a6daa1e931b038114c75             SP   12.99        perfumery   
4  00042b26cf59d7ce69dfabb4e55b4fd9             SP  199.90     garden_tools   

   review_score  order_date  
0             5  2017-09-13  
1             4  2017-04-26  
2             5  2018-01-14  
3             4  2018-08-08  
4             5  2017-02-04  


In [14]:
print(df[df.duplicated('order_id', keep=False)].sort_values('order_id').head(10))

                             order_id customer_state   price  \
71   002f98c0f7efd42638ed6100ca699b42             RS    8.99   
72   002f98c0f7efd42638ed6100ca699b42             RS   44.90   
127  005d9a5423d47281ac463a968b3936fb             SP   24.99   
128  005d9a5423d47281ac463a968b3936fb             SP   49.99   
201  0097f0545a302aafa32782f1734ff71c             SP  158.00   
202  0097f0545a302aafa32782f1734ff71c             SP  308.00   
258  00bcee890eba57a9767c7b5ca12d3a1b             DF  165.50   
259  00bcee890eba57a9767c7b5ca12d3a1b             DF  175.91   
396  01144cadcf64b6427f0a6580a3033220             SP   59.90   
397  01144cadcf64b6427f0a6580a3033220             SP   62.00   

                  category  review_score  order_date  
71          consoles_games             5  2017-08-04  
72                    toys             5  2017-08-04  
127                   baby             1  2017-10-18  
128                   toys             1  2017-10-18  
201           garden

In [15]:
print(df['review_score'].value_counts().sort_index())
print(df['category'].nunique())
print(df['customer_state'].nunique())
print(df[['price', 'review_score']].describe())

review_score
1     9900
2     3105
3     8122
4    18937
5    56977
Name: count, dtype: int64
71
27
              price  review_score
count  97041.000000  97041.000000
mean     124.309585      4.133397
std      186.718224      1.303541
min        0.850000      1.000000
25%       41.000000      4.000000
50%       78.990000      5.000000
75%      139.000000      5.000000
max     6735.000000      5.000000
